# Загрузка и сохранение данных

In [1]:
import sys

from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import config

In [2]:
from init_pipeline import init_experiment
init_experiment(project_root, config)

Инициализация пропущена: /Users/romansafronenkov/Documents/Projects/uplift_modeling_pipeline/artifacts/boosting_pipeline/status.json уже существует и init=True


## Логирование

In [3]:
import os

from src.utils.logger import setup_logging

LOG_DIR = project_root / 'artifacts' / config.general.experiment_name / 'log'
os.makedirs(LOG_DIR, exist_ok=True)

LOG_FILE = LOG_DIR / 'data_manipulation.txt'

_logger = setup_logging(LOG_FILE, 'data_manipulation')

/Users/romansafronenkov/Documents/Projects/uplift_modeling_pipeline/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Импорты

In [54]:
import shutil
import copy
import json
import random
import time

from IPython.display import clear_output

import pandas as pd
import numpy as np

import pyspark.sql.functions as F
from pyspark.sql.window import Window

from sklift.datasets import fetch_megafon

In [5]:
if config.general.load_venv_to_spark:
    os.environ['PYSPARK_PYTHON'] = './environment/bin/python'
    os.environ['PYSPARK_DRIVER_PYTHON'] = './environment/bin/python'

In [6]:
np.random.seed(config.general.seed)
random.seed(config.general.seed)

In [7]:
from src.utils.get_spark import get_conf, get_spark

conf = get_conf()

if config.general.load_venv_to_spark:
    conf.set('spark.archives', project_root / 'venv.tar.gz#environment')

In [10]:
spark = get_spark(conf, app_name=config.general.spark_session_name)

clear_output()
spark

# Загрузка датасета

In [23]:
data_dir = project_root / 'data'
os.makedirs(data_dir, exist_ok=True)

In [47]:
if config.general.create_dataset:
    # Загрузка датасета
    dataset = fetch_megafon()
    pdf = pd.concat([dataset.data, dataset.target, dataset.treatment], axis=1)
    
    temp_path = data_dir / "megafon_temp.csv"
    
    pdf.to_csv(temp_path, index=False)
    sdf = spark.read.option("header", "true").option("inferSchema", "true").csv(str(temp_path))
    
    final_path = data_dir / 'megafon_dataset.parquet'
    sdf.write.mode("overwrite").parquet(str(final_path))
    
    os.remove(str(temp_path))

-RECORD 0------------------------------
 X_1             | 5.983916871232271   
 X_2             | 2.9356989931769006  
 X_3             | 20.884441517959072  
 X_4             | -48.43777339584806  
 X_5             | -11.043025280502066 
 X_6             | -413.3841727181602  
 X_7             | -112.48871446560337 
 X_8             | 19.66138842179554   
 X_9             | -99.72146897068356  
 X_10            | 256.3139448817406   
 X_11            | -273.1617136432329  
 X_12            | 14.50271271055184   
 X_13            | 34.368866367991785  
 X_14            | -34.45689670960001  
 X_15            | 219.3091908232054   
 X_16            | -35.36025276351034  
 X_17            | 105.5435282137647   
 X_18            | -12.827160763977387 
 X_19            | 47.94737254924455   
 X_20            | 43.14810374268281   
 X_21            | 268.9417057340602   
 X_22            | -97.96349912935044  
 X_23            | 0.1423842859525586  
 X_24            | 0.4584347372469839  


In [58]:
if config.general.create_dataset:
    sdf = spark.read.parquet(str(final_path))
    sdf.show(1, vertical=True)

-RECORD 0------------------------------
 X_1             | 5.983916871232271   
 X_2             | 2.9356989931769006  
 X_3             | 20.884441517959072  
 X_4             | -48.43777339584806  
 X_5             | -11.043025280502066 
 X_6             | -413.3841727181602  
 X_7             | -112.48871446560337 
 X_8             | 19.66138842179554   
 X_9             | -99.72146897068356  
 X_10            | 256.3139448817406   
 X_11            | -273.1617136432329  
 X_12            | 14.50271271055184   
 X_13            | 34.368866367991785  
 X_14            | -34.45689670960001  
 X_15            | 219.3091908232054   
 X_16            | -35.36025276351034  
 X_17            | 105.5435282137647   
 X_18            | -12.827160763977387 
 X_19            | 47.94737254924455   
 X_20            | 43.14810374268281   
 X_21            | 268.9417057340602   
 X_22            | -97.96349912935044  
 X_23            | 0.1423842859525586  
 X_24            | 0.4584347372469839  


In [59]:
if config.general.create_dataset:
    sdf = sdf.withColumnRenamed('conversion', config.dataset.target_col)
    sdf = sdf.withColumnRenamed('treatment_group', config.dataset.treatment_col)
    
    sdf = sdf.withColumn(config.dataset.treatment_col,  F.when(F.col(config.dataset.treatment_col) == 'treatment', 1).otherwise(0))
    sdf.select(config.dataset.target_col, config.dataset.treatment_col).show()

+------+---------+
|target|treatment|
+------+---------+
|     0|        1|
|     0|        1|
|     0|        0|
|     0|        0|
|     1|        0|
|     1|        0|
|     1|        0|
|     0|        1|
|     1|        1|
|     0|        1|
|     0|        0|
|     1|        0|
|     0|        1|
|     0|        0|
|     0|        1|
|     0|        0|
|     0|        1|
|     0|        1|
|     0|        1|
|     0|        1|
+------+---------+
only showing top 20 rows



In [63]:
if config.general.create_dataset:
    # для демонстрации создадим фейк колонку с датой
    dates = [f"2025-{i:02d}-01" for i in range(1, 11)]  # ['2025-01-01', ..., '2025-10-01']
    dates_array = F.array([F.lit(d) for d in dates])
    
    window_spec = Window.partitionBy(config.dataset.target_col, config.dataset.treatment_col).orderBy(F.rand(seed=config.general.seed))
    
    sdf = sdf.withColumn("rn", F.row_number().over(window_spec))
    
    # Назначаем дату по модулю 10 (чередование) – за счёт случайного порядка внутри группы
    # каждая дата попадёт примерно в равное количество строк.
    sdf = sdf.withColumn(
        config.dataset.date_col,
        F.element_at(dates_array, (F.col("rn") - 1) % 10 + 1)
    )
    
    # 6. Удаляем временную колонку rn
    sdf = sdf.drop("rn")
    
    # 7. Проверяем, что получилось
    sdf.groupBy(config.dataset.date_col, config.dataset.treatment_col).agg(F.count(config.dataset.target_col)).toPandas()

,date,treatment,count(target)
0,2025-10-01,0,29962
1,2025-07-01,1,30037
2,2025-10-01,1,30036
3,2025-05-01,0,29963
4,2025-04-01,1,30037
5,2025-09-01,0,29962
6,2025-02-01,0,29964
7,2025-02-01,1,30037
8,2025-05-01,1,30037
9,2025-06-01,0,29963


In [66]:
if config.general.create_dataset:
    unique_dates = sdf.select(config.dataset.date_col).distinct().collect()
    unique_dates = sorted([row[config.dataset.date_col] for row in unique_dates])
    unique_dates

['2025-01-01',
 '2025-02-01',
 '2025-03-01',
 '2025-04-01',
 '2025-05-01',
 '2025-06-01',
 '2025-07-01',
 '2025-08-01',
 '2025-09-01',
 '2025-10-01']

In [67]:
if config.general.create_dataset:
    train_dataset = sdf.filter(~(F.col(config.dataset.date_col) == unique_dates[-1]))
    
    stratification_columns = [config.dataset.date_col, config.dataset.treatment_col, config.dataset.target_col]
    
    train_dataset = train_dataset.withColumn('_rand', F.rand(seed=config.general.seed))
    window_spec = Window.partitionBy(*stratification_columns).orderBy('_rand')
    train_dataset = train_dataset.withColumn('_rn', F.row_number().over(window_spec))
    
    train_dataset = train_dataset.withColumn('_cnt', F.count('*').over(Window.partitionBy(*stratification_columns)))
    train_dataset = (
        train_dataset
        .withColumn('is_oos',
                   F.when(F.col('_rn') <= F.ceil(F.col('_cnt') * config.dataset.oos_fraction), True).otherwise(False)
                   )
    )
    
    oos_dataset = train_dataset.filter(F.col('is_oos') == True).drop('_rand', '_rn', '_cnt', 'is_oos')
    train_dataset = train_dataset.filter(F.col('is_oos') == False).drop('_rand', '_rn', '_cnt', 'is_oos')
    
    _logger.info(f'Train len: {train_dataset.count()}\nOOS len: {oos_dataset.count()}')
    
    oot_dataset = sdf.filter(F.col(config.dataset.date_col) == unique_dates[-1])
    _logger.info(f'OOT len: {oot_dataset.count()}')
    
    train_dataset.write.mode("overwrite").parquet(str(data_dir / 'train_dataset.parquet'))
    oos_dataset.write.mode("overwrite").parquet(str(data_dir / 'oos_dataset.parquet'))
    oot_dataset.write.mode("overwrite").parquet(str(data_dir / 'oot_dataset.parquet'))

[2026-09-01 15:25:28,332] - [data_manipulation] - [INFO] - Train len: 458976
OOS len: 81026
[2026-09-01 15:25:28,701] - [data_manipulation] - [INFO] - OOT len: 59998


## Проверка корректности

In [68]:
if config.general.check_target_validity:
    train_dataset = spark.read.parquet(str(data_dir / 'train_dataset.parquet'))
    oos_dataset = spark.read.parquet(str(data_dir / 'oos_dataset.parquet'))
    oot_dataset = spark.read.parquet(str(data_dir / 'oot_dataset.parquet'))
    
    dataset_full = spark.read.parquet(str(data_dir / 'megafon_dataset.parquet'))

In [69]:
if config.general.check_target_validity:
    len_train = train_dataset.count()
    len_oos = oos_dataset.count()
    len_oot = oot_dataset.count()
    len_full = dataset_full.count()
    
    assert len_train + len_oos + len_oot == len_full, 'Не совпадает'

In [71]:
if config.general.check_target_validity:
    train_dataset_pd = train_dataset.select(config.dataset.date_col, config.dataset.treatment_col, config.dataset.target_col).toPandas()
    oos_dataset_pd = oos_dataset.select(config.dataset.date_col, config.dataset.treatment_col, config.dataset.target_col).toPandas()
    oot_dataset_pd = oot_dataset.select(config.dataset.date_col, config.dataset.treatment_col, config.dataset.target_col).toPandas()

In [74]:
if config.general.check_target_validity:
    uplift_rate = (
        train_dataset_pd
        .groupby([config.dataset.date_col, config.dataset.treatment_col], as_index=False)[config.dataset.target_col].mean()
        .sort_values([config.dataset.date_col, config.dataset.treatment_col], ascending=False)
    )
    
    uplift_rate = uplift_rate.groupby(config.dataset.date_col)[config.dataset.target_col].apply(lambda x: x.iloc[0] - x.iloc[1])
    _logger.info(f'Train uplift rate:\n{uplift_rate}')

[2026-09-01 15:32:30,597] - [data_manipulation] - [INFO] - Train uplift rate:
date
2025-01-01    0.049506
2025-02-01    0.049506
2025-03-01    0.049506
2025-04-01    0.049506
2025-05-01    0.049538
2025-06-01    0.049538
2025-07-01    0.049538
2025-08-01    0.049538
2025-09-01    0.049531
Name: target, dtype: float64


In [75]:
if config.general.check_target_validity:    
    uplift_rate = (
        oos_dataset_pd
        .groupby([config.dataset.date_col, config.dataset.treatment_col], as_index=False)[config.dataset.target_col].mean()
        .sort_values([config.dataset.date_col, config.dataset.treatment_col], ascending=False)
    )
    
    uplift_rate = uplift_rate.groupby(config.dataset.date_col)[config.dataset.target_col].apply(lambda x: x.iloc[0] - x.iloc[1])
    _logger.info(f'OOS uplift rate:\n{uplift_rate}')

[2026-09-01 15:32:44,248] - [data_manipulation] - [INFO] - OOS uplift rate:
date
2025-01-01    0.049484
2025-02-01    0.049484
2025-03-01    0.049484
2025-04-01    0.049484
2025-05-01    0.049484
2025-06-01    0.049484
2025-07-01    0.049484
2025-08-01    0.049484
2025-09-01    0.049535
Name: target, dtype: float64


In [76]:
if config.general.check_target_validity:
    uplift_rate = (
        oot_dataset_pd
        .groupby([config.dataset.date_col, config.dataset.treatment_col], as_index=False)[config.dataset.target_col].mean()
        .sort_values([config.dataset.date_col, config.dataset.treatment_col], ascending=False)
    )
    
    uplift_rate = uplift_rate.groupby(config.dataset.date_col)[config.dataset.target_col].apply(lambda x: x.iloc[0] - x.iloc[1])
    _logger.info(f'OOT uplift rate:\n{uplift_rate}')

[2026-09-01 15:32:53,942] - [data_manipulation] - [INFO] - OOT uplift rate:
date
2025-10-01    0.049531
Name: target, dtype: float64


In [77]:
with open(project_root / 'artifacts' / config.general.experiment_name / 'status.json', 'w') as f:
    status = {
        'init': True,
        'dataset': True,
        'features': False,
        'choose_model': False,
        'optimization': False,
        'fitting': False
    }
    json.dump(status, f)

In [78]:
spark.stop()

In [ ]:
os._exit(00)